This code defines a **Flask blueprint** for handling customer-related functionality in a food delivery platform LIEFERSPATZ. Below is a detailed explanation of each section:

---

## **Imports**
```python
from flask import Blueprint, render_template, request, redirect, url_for, flash, session
from db import get_db_connection
from werkzeug.security import generate_password_hash
from datetime import datetime
import sqlite3
```
- `Blueprint`: Allows modular organization of the app by grouping related routes together.
- `render_template`: Renders HTML templates.
- `request`: Handles incoming HTTP requests.
- `redirect`, `url_for`: Redirects users to different routes.
- `flash`: Displays temporary messages to users.
- `session`: Stores user data across requests.
- `get_db_connection`: A function (assumed to be in `db.py`) to connect to the SQLite database.
- `generate_password_hash`: Encrypts user passwords.
- `datetime`: Handles timestamps.
- `sqlite3`: Provides database connectivity.

---

## **Blueprint Creation**
```python
customer_bp = Blueprint('customer', __name__)
```
- Defines a `Blueprint` named `'customer'`, which groups all customer-related routes.

---

## **Utility Function: `array_merge`**
```python
def array_merge( first_array, second_array ):
    if isinstance(first_array, list) and isinstance(second_array, list): 
        return first_array + second_array
    elif isinstance(first_array, dict) and isinstance(second_array, dict): 
        return dict(list(first_array.items()) + list(second_array.items() ))
    elif isinstance(first_array, set) and isinstance(second_array, set): 
        return first_array.union(second_array)
    return False
```
- Merges two lists, dictionaries, or sets:
  - Lists: Combines elements.
  - Dicts: Merges key-value pairs (keys in `second_array` overwrite `first_array`).
  - Sets: Returns the union of both sets.
  - Returns `False` if types don't match.

---

## **1. Customer Registration Route**
```python
@customer_bp.route('/register/customer', methods=['GET', 'POST'])
def register_customer():
```
- Registers a new customer.
- Handles both `GET` (display form) and `POST` (submit form) requests.

```python
if request.method == 'POST':
```
- If the form is submitted:

```python
    first_name = request.form['first_name']
    last_name = request.form['last_name']
    address = request.form['address']
    zip_code = request.form['zip_code']
    phone_number = request.form['phone_number']
    password = generate_password_hash(request.form['password'])
```
- Retrieves form inputs and encrypts the password.

### **Validation**
```python
    if not zip_code.isdigit():
        flash('Please provide a valid ZIP Code', 'danger')
        return render_template('register_customer.html')
    
    if not phone_number.isdigit():
        flash('Please provide a valid phone number', 'danger')
        return render_template('register_customer.html')
```
- Ensures ZIP and phone number are numeric.

### **Check for Existing Phone Number**
```python
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('SELECT 1 FROM customers WHERE PhoneNumber = ?', (phone_number,))
    existing_customer = cursor.fetchone()
```
- Checks if the phone number is already in use.

```python
    if existing_customer:
        flash('Phone Number is already in use, please choose a different one.', 'danger')
        conn.close()
        return render_template('register_customer.html')
```
- If the number exists, displays an error message.

### **Insert Customer Data**
```python
    cursor.execute('''
        INSERT INTO customers (FirstName, LastName, Address, ZipCode, PhoneNumber, Password, CreatedAt)
        VALUES (?, ?, ?, ?, ?, ?, ?)
    ''', (first_name, last_name, address, zip_code, phone_number, password, datetime.now()))
    conn.commit()
    conn.close()
```
- Inserts customer details into the database.

```python
    flash('Customer registration successful!', 'success')
    return redirect(url_for('auth.login'))
```
- Displays a success message and redirects to the login page.

---

## **2. Customer Dashboard**
```python
@customer_bp.route('/customer/dashboard')
def customer_dashboard():
```
- Displays the customer dashboard.

```python
    if 'customer' not in session:
        return redirect(url_for('auth.login'))
```
- Redirects users who are not logged in.

```python
    if 'shoppingcart' in session:
        flash('You May Only Choose One Restaurant To Order From, Empty Shopping Cart First.', 'danger')
        return redirect(url_for('customer.itemorder'))
```
- Prevents customers from ordering from multiple restaurants.

### **Fetch Nearby Restaurants**
```python
    conn = get_db_connection()
    cursor = conn.cursor()
    customer_data = session['customer']
    cursor.execute('SELECT * FROM restaurants WHERE RestaurantID IN (SELECT RestaurantID FROM delivery_zip_codes WHERE ZipCode = ?)', (customer_data['ZipCode'],))
    restaurantsclose = cursor.fetchall()
    conn.close()
```
- Fetches restaurants that deliver to the user's ZIP code.

```python
    return render_template('dashboard_customer.html', user=customer_data, restaurantsclose=restaurantsclose)
```
- Renders the dashboard template.

---

## **3. Item Ordering**
```python
@customer_bp.route('/customer/itemorder', methods=['GET', 'POST'])
def itemorder():
```
- Displays menu items from a selected restaurant.

```python
    if 'customer' not in session:
        return redirect(url_for('auth.login'))
```
- Ensures the user is logged in.

### **User Selects a Restaurant**
```python
    if request.method == 'POST':
        customer_data = session['customer']
        chosenID = request.form['selectedID']
        session['chosenrestID'] = chosenID
```
- Stores the selected restaurant in the session.

### **Fetch Restaurant Details and Menu**
```python
        cursor.execute('SELECT * FROM restaurants WHERE RestaurantID = ?', (chosenID,)) 
        truerestaurantchosen = cursor.fetchone()
        cursor.execute('SELECT * FROM Items WHERE RestaurantID = ? ORDER BY Category, Name', (chosenID,))
        trueitemschosen = cursor.fetchall()
```
- Retrieves restaurant details and menu items.

```python
        return render_template('itemorder.html', user=customer_data, itemschosen=trueitemschosen, restaurantchosen=truerestaurantchosen)
```
- Renders the menu page.

---

## **4. Adding Items to Cart**
```python
@customer_bp.route('/customer/addtocart', methods=['GET', 'POST'])
def addtocart():
```
- Adds items to the shopping cart.

```python
    if 'customer' not in session:
        return redirect(url_for('auth.login'))
```
- Ensures user is logged in.

```python
    if request.method == 'POST':
        quantity = int(request.form['productquantity'])
        itemtoadd = int(request.form['chosenItemID'])
```
- Gets item and quantity from the form.

```python
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute('SELECT * FROM Items WHERE ItemID = ?', (itemtoadd,))
        itemrow = cursor.fetchone()
```
- Fetches the item details.

```python
        itemDict = { str(itemrow['ItemID']) : {'Name' : itemrow['Name'], 'ItemToAdd' : itemrow['ItemID'], 'Quantity' : quantity, 'Price' : itemrow['Price'], 'TotalPrice' : itemrow['Price']*quantity } }
```
- Creates a dictionary for the item.

```python
        if 'shoppingcart' in session:
            if itemrow['ItemID'] in session['shoppingcart']:
                oldquantity = session['shoppingcart'][key]['Quantity']
                total_quantity = oldquantity + quantity
                session['shoppingcart'][key]['Quantity'] = total_quantity
                session['shoppingcart'][key]['TotalPrice'] = total_quantity * itemDict['Price']
            else:
                session['shoppingcart'] = array_merge(session['shoppingcart'], itemDict)
```
- Updates cart quantity if item exists, otherwise adds it.

```python
        conn.close()
        return redirect(url_for('customer.itemorder'))
```
- Saves changes and redirects back.

Here’s a detailed line-by-line explanation for each of the requested routes:

---

### **5. Emptying the Cart (`/customer/empty`)**
```python
@customer_bp.route('/customer/empty')
```
- This defines a new route `/customer/empty` within the `customer_bp` blueprint.
- It allows customers to empty their shopping cart.

```python
def empty_cart():
```
- Defines the function that handles the request.

```python
    if 'shoppingcart' in session:
```
- Checks if the `shoppingcart` key exists in the session.
- The session stores temporary data like shopping cart contents.

```python
        session.pop('shoppingcart')
```
- Removes the `shoppingcart` from the session.
- This effectively empties the cart.

```python
        session['total_price'] = 0
        session['total_quantity'] = 0
```
- Resets the total price and total quantity to 0 since the cart is now empty.

```python
    return redirect(url_for('customer.itemorder'))
```
- Redirects the user back to the item ordering page.

---

### **6. Deleting Items from Cart (`/customer/deleteproduct`)**
```python
@customer_bp.route('/customer/deleteproduct', methods=['GET', 'POST'])
```
- Defines the `/customer/deleteproduct` route.
- Accepts both `GET` and `POST` requests.

```python
def deleteproduct():
```
- Defines the function for handling item deletion.

```python
    if request.method == 'POST':
```
- Ensures that the request is of type `POST`.

```python
        itemtodelete = request.form['deleteItemID']
```
- Retrieves the `deleteItemID` from the submitted form data.
- This is the ID of the item that the customer wants to delete.

```python
        totalprice = 0
        totalquantity = 0
```
- Initializes variables for recalculating the total price and quantity.

```python
        session.modified = True
```
- Marks the session as modified, ensuring changes are saved.

```python
        for item in session['shoppingcart'].items(): 
```
- Iterates through the shopping cart dictionary.

```python
            if item[0] == itemtodelete:
```
- Finds the item that matches the `deleteItemID`.

```python
                session['shoppingcart'].pop(item[0], None)
```
- Removes the item from the cart.

```python
                if 'shoppingcart' in session:
```
- Checks if the cart still contains items after deletion.

```python
                    for key, value in session['shoppingcart'].items():
```
- Iterates through the remaining items in the cart.

```python
                        singlequantity = session['shoppingcart'][key]['Quantity']
                        singleprice = session['shoppingcart'][key]['TotalPrice']
```
- Retrieves quantity and total price for each remaining item.

```python
                        totalprice += singleprice
                        totalquantity += singlequantity
```
- Updates the overall total price and quantity.

```python
                break 
```
- Exits the loop after deleting the item.

```python
        if totalquantity == 0:
            session.pop('shoppingcart')
```
- If no items are left in the cart, remove the `shoppingcart` from the session.

```python
        else:
            session['total_quantity'] = totalquantity
            session['total_price'] = totalprice
```
- If items remain, update the total price and quantity.

```python
    return redirect(url_for('customer.itemorder'))
```
- Redirects the user back to the item ordering page.

---

### **7. Handling Payments (`/customer/handle_payment/<action>`)**

**paymentconfirm.html first checks if the session[total_price'] <= customer['balance']**
- If no then it sends action="invalid" to customer/handle_payment/action=invalid route in customer.py
- If yes then it sends the data to socket.emit(join) and socket.emit(send_payment) route in websockets.py, websockets.py's send_payment method then emits "payment_received"
    - then any page on the restaurant side recieves a notification "Accept" or "Decline" as they constantly listen for method socket.on('payment_received') and then socket.emit("restaurant_reply") with "Payment Accepted" or "Payment Declined"
    - the websockets.py's restaurant_reply then emit('restaurant_ack') with data, 
        - which is then being listened by paymentconfirm.html page, which checks if we have "Payment Accepted" then sends action = "Accept" to customer/handle_payment/action=accept route in customer.py
        - Or in case of "Payment Declined" sends action = "Decline" to customer/handle_payment/action=decline route in customer.py **

```python
@customer_bp.route('/customer/handle_payment/<action>', methods=['GET', 'POST'])
```
- Defines a dynamic route where `<action>` can be either `accept` or `invalid`.
- Accepts both `GET` and `POST` requests.

```python
def handle_payment(action):
```
- Defines the function to process payment.

```python
    if 'customer' not in session:
        return redirect(url_for('auth.login'))
```
- Ensures that only logged-in customers can access this route.

```python
    customer_data = session['customer']
```
- Retrieves customer data from the session.

```python
    conn = get_db_connection()
    cursor = conn.cursor()
```
- Establishes a connection to the database.

```python
    cursor.execute('SELECT Balance FROM customers WHERE CustomerID = ?', (customer_data['CustomerID'],))
    balance = cursor.fetchone()
```
- Fetches the customer’s balance from the database.

```python
    if action == 'invalid':
```
- Checks if the action is `invalid` (indicating insufficient funds).

```python
        flash("Insufficient Balance Amount.", 'danger')
```
- Displays an error message.

```python
        session.pop('shoppingcart')
        session['total_price'] = 0
        session['total_quantity'] = 0
```
- Clears the shopping cart.

```python
        conn.close()
        return redirect(url_for('customer.customer_dashboard'))
```
- Closes the database connection and redirects the user.

```python
    else:
        NotesToAdd = request.form.get('notestoadd', '')
        status = 'InProcess' if action == 'accept' else 'Rejected'
```
- Gets additional order notes and sets the order status.

```python
        restaurant_money = 0.00
        liefer_money = 0.00
        if action == 'accept':
```
- Initializes the restaurant’s share and Lieferspatz’s commission.

```python
            total_price = session['total_price']
            restaurant_money = round(total_price * 0.85, 2)
            liefer_money = round(total_price * 0.15, 2)
```
- Splits the total payment (85% to the restaurant, 15% to Lieferspatz).

```python
        cursor.execute('''
            INSERT INTO Orders (CustomerID, RestaurantID, Notes, TotalPrice, Status, CreatedAt, RestaurantMoney, LieferMoney)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ''', (customer_data['CustomerID'], session['chosenrestID'], NotesToAdd, session['total_price'], status, datetime.now(), restaurant_money, liefer_money))
```
- Inserts a new order into the `Orders` table.

```python
        OrderID = cursor.lastrowid
```
- Retrieves the ID of the newly inserted order.

```python
        for key, value in session['shoppingcart'].items():
```
- Iterates through the cart to insert items into `OrderItems`.

```python
            cursor.execute('INSERT INTO OrderItems (OrderID, ItemID, Quantity, Price) VALUES (?, ?, ?, ?)', 
                           (OrderID, value['ItemToAdd'], value['Quantity'], value['Price']))
```
- Inserts each ordered item into the `OrderItems` table.

```python
        if action == 'accept':
```
- Checks if the order was accepted.

```python
            newbalance = balance[0] - session['total_price']
            cursor.execute('UPDATE customers SET Balance = ? WHERE CustomerID = ?', (newbalance, customer_data['CustomerID']))
            session['customer']['Balance'] = newbalance
```
- Deducts the total price from the customer’s balance.

```python
            cursor.execute('SELECT Balance FROM restaurants WHERE RestaurantID = ?', (session['chosenrestID'],))
            restaurant_balance = cursor.fetchone()[0]
            new_restaurant_balance = restaurant_balance + restaurant_money
            cursor.execute('UPDATE restaurants SET Balance = ? WHERE RestaurantID = ?', (new_restaurant_balance, session['chosenrestID']))
```
- Updates the restaurant’s balance.

```python
        conn.commit()
        conn.close()
```
- Saves and closes the database connection.

```python
        flash("Payment Accepted and Successful." if action == 'accept' else "Payment Declined by Restaurant.", 'success' if action == 'accept' else 'danger')
```
- Displays a success or failure message.

```python
        session.pop('shoppingcart', None)
        session['total_price'] = 0
        session['total_quantity'] = 0
```
- Clears the cart.

```python
        return redirect(url_for('customer.customer_dashboard'))
```
- Redirects the customer.

---

### **8. Viewing Past Orders (`/customer/past_orders`)**
This route fetches and displays a customer’s past orders, including item details.

### **Viewing Past Orders (`/customer/past_orders`) – Line-by-Line Explanation**  

This route allows a logged-in customer to view their past orders, including details of the items ordered, order status, and payment details.

### **1. Route Definition**
```python
@customer_bp.route('/customer/past_orders', methods=['GET'])
```
- Defines a new route `/customer/past_orders` within the `customer_bp` blueprint.
- It only supports `GET` requests, meaning no data is sent from the user—only retrieval.


### **2. Function Definition**
```python
def past_orders():
```
- Defines the function that handles the request for past orders.

```python
    if 'customer' not in session:
        return redirect(url_for('auth.login'))
```
- Checks if a customer is logged in. If not, they are redirected to the login page.

### **3. Database Connection**
```python
    customer_data = session['customer']
    conn = get_db_connection()
    cursor = conn.cursor()
```
- Retrieves customer data from the session (such as `CustomerID`).
- Establishes a database connection and creates a cursor to execute SQL queries.

### **4. Fetching Past Orders**
```python
    cursor.execute('''
        SELECT o.OrderID, o.RestaurantID, r.RestaurantName, o.TotalPrice, o.Status, o.CreatedAt
        FROM Orders o
        JOIN Restaurants r ON o.RestaurantID = r.RestaurantID
        WHERE o.CustomerID = ?
        ORDER BY o.CreatedAt DESC
    ''', (customer_data['CustomerID'],))
```
- Selects past orders made by the customer.
- Joins the `Orders` table with the `Restaurants` table to get the restaurant's name.
- Orders the results by `CreatedAt` in descending order (most recent first).

```python
    past_orders = cursor.fetchall()
```
- Fetches all matching orders from the database and stores them in a list.

### **5. Fetching Order Items for Each Order**
```python
    order_details = {}
    for order in past_orders:
```
- Initializes an empty dictionary `order_details` to store items for each order.
- Iterates over each order retrieved from the database.

```python
        cursor.execute('''
            SELECT oi.ItemID, i.ItemName, oi.Quantity, oi.Price
            FROM OrderItems oi
            JOIN Items i ON oi.ItemID = i.ItemID
            WHERE oi.OrderID = ?
        ''', (order['OrderID'],))
```
- Fetches all items that belong to a particular order.
- Joins the `OrderItems` table with `Items` to get the item names.

```python
        order_items = cursor.fetchall()
```
- Retrieves the items for the order.

```python
        order_details[order['OrderID']] = order_items
```
- Stores the items in the `order_details` dictionary, using the `OrderID` as the key.

### **6. Closing Connection and Rendering the Orders Page**
```python
    conn.close()
```
- Closes the database connection.

```python
    return render_template('customer/past_orders.html', past_orders=past_orders, order_details=order_details)
```
- Passes the `past_orders` and `order_details` data to the template `past_orders.html`.
- The HTML template will display the order history with order details.

---

### **Final Output**
- When a logged-in customer visits `/customer/past_orders`, they see:
  - A list of their past orders, including restaurant name, order date, total price, and order status.
  - The items within each order (item name, quantity, and price).
